**Coding along - Data Analysis and visualization - Day 3**

1. Look
2. Find the problems
3. Fix them one at a time
4. Never change the original
5. Write the steps as code

In [1]:
import pandas as pd

orders_raw = pd.read_csv('data/orders.csv')
orders = orders_raw.copy() # When cleaning a table we never write changes to the original.

orders.shape

(825, 10)

**First look**

In [ ]:
orders.head(10)

In [ ]:
orders.info()

## Test I: Spaces and capital letters

In [ ]:
orders['category'].unique()

In [ ]:
orders['category'].value_counts()

In [7]:
print((orders['category'] == 'Clothing').sum())

128


.str

In [11]:
print(orders['category'].nunique())
print(orders['category'].str.strip().nunique())
print(orders['category'].str.strip().str.lower().nunique())

38
20
7


In [12]:
print(orders['category'].str.strip().str.lower().value_counts())

category
clothing           180
electronics        151
sports             148
shoes              143
home & garden      114
books               70
home and garden     19
Name: count, dtype: int64


In [13]:
orders['category'] = (
    orders['category']
    .str.strip()
    .str.lower()
    .replace('home and garden', 'home & garden') # Replaces and merges the categories.
    .str.title()
)

orders['category'].value_counts()

category
Clothing         180
Electronics      151
Sports           148
Shoes            143
Home & Garden    133
Books             70
Name: count, dtype: int64

In [14]:
orders['product'].nunique()

24

**Text II: yes/no and prices that are text**

In [15]:
orders['express'].value_counts() # Before the cleanup, we get 6 different values for something that should be bool.

express
no     300
No     166
yes    152
NO      88
Yes     81
y       38
Name: count, dtype: int64

In [16]:
orders['express'] = orders['express'].str.lower().map({'yes':True, 'y':True, 'no':False})

print(orders['express'].isna().sum())
orders['express'].value_counts()

0


express
False    554
True     271
Name: count, dtype: int64

In [17]:
orders['unit_price'].head(8).tolist()

['1 199,00 kr',
 '349 kr',
 '1 039,20 kr',
 '249 kr',
 '1299 kr',
 '349 kr',
 '199 kr',
 '1 039,20 kr']

In [ ]:
orders['unit_price'].astype(float) # This won't work, since, first, we have to clean the text from, for example "kr".

In [20]:
price = orders['unit_price'].str.strip()
print(price.head(3).tolist())

price = price.str.replace(' kr', '')
print(price.head(3).tolist())

price =  price.str.replace(' ', '')
print(price.head(3).tolist())

price = price.str.replace(',', '.')
print(price.head(3).tolist())

# Here the teacher shows how to do it in one line by using regex.

['1 199,00 kr', '349 kr', '1 039,20 kr']
['1 199,00', '349', '1 039,20']
['1199,00', '349', '1039,20']
['1199.00', '349', '1039.20']


In [22]:
orders['unit_price'] = price.astype(float)

print(orders['unit_price'].dtype)
orders['unit_price'].describe().round(2)

float64


count     825.00
mean      429.06
std       374.35
min        59.00
25%       149.00
50%       249.00
75%       699.00
max      1499.00
Name: unit_price, dtype: float64

## Dates: pd.to_datetime()

In [23]:
pd.to_datetime(orders['order_date'])

ValueError: time data "unknown" doesn't match format "%Y-%m-%d". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [24]:
dates = pd.to_datetime(orders['order_date'], errors='coerce') # becomes NaT (not a time)

print(orders['order_date'].isna().sum())    # missing before
print(dates.isna().sum())                   # missing after

orders.loc[dates.isna(), ['order_id', 'order_date']]

1
4


,order_id,order_date
51,ORD-20048,unknown
100,ORD-20096,NaN
537,ORD-20523,2026-02-30
784,ORD-20764,2026-04-31


In [25]:
orders['order_date'] = dates

print(orders['order_date'].dtype)
print(orders['order_date'].min())
print(orders['order_date'].max())
print(orders['order_date'].max() - orders['order_date'].min())


datetime64[us]
2026-01-05 00:00:00
2026-06-28 00:00:00
174 days 00:00:00


orders per month using .dt

In [26]:
orders['order_date'].dt.to_period('M').value_counts().sort_index()



order_date
2026-01    136
2026-02    135
2026-03    140
2026-04    130
2026-05    162
2026-06    118
Freq: M, Name: count, dtype: int64

## Duplicates: duplicated()

In [27]:
print(orders.duplicated().sum()) # Here we get 25 duplicates.

25


In [28]:
orders[orders.duplicated(keep=False)].head(6)

,order_id,customer_id,order_date,product,category,unit_price,quantity,express,coupon_code,gift_message
26,ORD-20027,C0115,2026-01-09,Fleece sweater,Clothing,699.0,1,False,NaN,NaN
27,ORD-20027,C0115,2026-01-09,Fleece sweater,Clothing,699.0,1,False,NaN,NaN
34,ORD-20034,C0285,2026-01-10,Plant pot,Home & Garden,89.0,1,False,NaN,NaN
35,ORD-20034,C0285,2026-01-10,Plant pot,Home & Garden,89.0,1,False,NaN,NaN
40,ORD-20039,C0128,2026-01-10,Bike light,Electronics,199.2,2,True,NaN,NaN
41,ORD-20039,C0128,2026-01-10,Bike light,Electronics,199.2,2,True,NaN,NaN


In [29]:
print(orders_raw.duplicated().sum()) # Looking for duplicates in the original untouched data.

20


In [30]:
print(orders.duplicated(subset='order_id').sum())

25


In [31]:
total_with = (orders['unit_price'] * orders['quantity']).sum()

orders = orders.drop_duplicates().reset_index(drop=True)

total_without = (orders['unit_price'] * orders['quantity']).sum()

print(orders.shape)
print(round(total_with, 2))
print(round(total_without, 2))
print(round(total_with - total_without, 2))


(800, 10)
591385.2
572783.8
18601.4


## Drop what we do not need: drop()

In [32]:
orders = orders.drop(columns='gift_message')

orders.shape

(800, 9)

**Which category brought in the most money?**

In [33]:
orders['total'] = orders['unit_price'] * orders['quantity']

orders.groupby('category')['total'].sum().round().sort_values(ascending=False)



category
Shoes            207382.0
Clothing         153808.0
Electronics       93745.0
Sports            56565.0
Home & Garden     44604.0
Books             16680.0
Name: total, dtype: float64

In [35]:
monthly = orders.groupby(orders['order_date'].dt.to_period('M'))['total'].sum()

print(monthly.round(0))
print(round(monthly.sum(), 2))
print(round(orders['total'].sum(), 2))

order_date
2026-01    106577.0
2026-02     93654.0
2026-03     91064.0
2026-04     74254.0
2026-05    126013.0
2026-06     78478.0
Freq: M, Name: total, dtype: float64
570040.8
572783.8


## After this comment, the teacher gives all of the cleaning steps in one note.

In [ ]:
orders_clean.to_csv('data/orders_clean.csv', index=False) # Orders clean is created with a code snippet from the teacher.